In [53]:
!pip3 install flask

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 102 kB 1.2 MB/s eta 0:00:01
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.


In [46]:
import pandas as pd
import requests
import json
import time
import csv
import io
import Levenshtein
from urllib.parse import urlparse
from flask import Flask, jsonify
from requests.auth import HTTPBasicAuth

In [47]:
def fetch_data_csv(url):
    # Make the request
    response = requests.get(url)
    if response.status_code == 200:
        # Return the CSV data
        csv_data = response.text
        csv_reader = csv.reader(io.StringIO(csv_data))

        # Convert CSV to a list of rows
        data = [row for row in csv_reader]
        return data
    else:
        print(f"Failed to fetch data. Status code: {response.status_code}. Reason: {response.reason}")
        
        return []

# Function to fetch exchanges with pagination
def fetch_exchanges(url):
    page = 1
    all_exchanges = []
    
    while True:
        # Add the page parameter to the request
        params = {"per_page": 250, "page": page} 
        try:
            # Make the request
            response = requests.get(url, params=params)
            # Check if the request was successful
            if response.status_code == 200:
                data = response.json()
                # If no data is returned, stop the loop
                if not data:
                    break
                # Add the fetched data to the list
                all_exchanges.extend(data)
                print(f"Fetched page {page} with {len(data)} exchanges.")
                # Move to the next page
                page += 1
                # Add a delay to avoid hitting the rate limit
                time.sleep(1) 
    
            elif response.status_code == 429:
                # Handle rate limit error
                print("Rate limit exceeded. Waiting for 10 seconds before retrying...")
                time.sleep(60)  # Wait 60 seconds before retrying
            
            else:
                # Handle other errors
                print(f"Failed to fetch data. Status code: {response.status_code}. Reason: {response.reason}")
                break
        
        except requests.exceptions.RequestException as e:
            # Handle network errors
            print(f"Network error: {e}")
            break
    
    return all_exchanges

# Function to fetch exchanges from DeFi Llama
def fetch_exchanges_II(url):
    # Make the request
    response = requests.get(url)
    if response.status_code == 200:
        # Return the JSON data
        return response.json() 
    else:
        print(f"Failed to fetch data. Status code: {response.status_code}. Reason: {response.reason}")
        
        return []

# Function to fetch the blacklist url
def blacklist_toMap(items):
    # Create a list to store the name-url mappings
    blacklist_map = []

    # Process each URL in the blacklist
    for url in items:
        # Extract the text before the domain extension
        name = url.split('.')[0]
        # Create a dictionary for the name and URL
        blacklist_map.append({'name': name, 'url': url})

    return blacklist_map


def fetch_bank_institutions(auth_url, institutions_url, client_id, client_secret):
    # authenticate and get access token
    auth_response = requests.post(auth_url, auth=HTTPBasicAuth(client_id, client_secret), data={"grant_type": "client_credentials"})
    
    if auth_response.status_code != 200:
        return f"Authentication failed: {auth_response.text}"
    
    access_token = auth_response.json().get("access_token")
    headers = {"Authorization": f"Bearer {access_token}"}
    
    # fetch financial institutions
    response = requests.get(institutions_url, headers=headers)
    
    if response.status_code != 200:
        return f"Failed to fetch institutions: {response.text}"
    
    institutions = response.json()
    return [{"id": inst["id"], "name": inst["name"], "countries": inst["countries"]} for inst in institutions]




# function to extract domain
extract_domain = lambda url: urlparse(url if "://" in url else f"https://{url}").netloc.replace("www.", "")

# Define the isTyposquatting function
def isTyposquatting(legit_domains, scam_domain, threshold=2):
    for legit_domain in legit_domains:
        if Levenshtein.ratio(legit_domain, scam_domain) <= threshold:
            return True
    return False

def format_domain(domain):
    if not domain.startswith(('https://', 'http://', 'www.')):
        return 'https://' + domain
    return domain


In [48]:
# fetch centralized exchanges from coingecko
cex_url = "https://api.coingecko.com/api/v3/exchanges"
gecko_data = fetch_exchanges(cex_url)
# view the first item of the data
gecko_data[0]

Fetched page 1 with 250 exchanges.
Fetched page 2 with 250 exchanges.
Fetched page 3 with 250 exchanges.
Fetched page 4 with 227 exchanges.


{'id': 'binance',
 'name': 'Binance',
 'year_established': 2017,
 'country': 'Cayman Islands',
 'description': 'One of the world’s largest cryptocurrency exchanges by trading volume, offering a wide range of services including spot, futures, and staking options.',
 'url': 'https://www.binance.com/',
 'image': 'https://coin-images.coingecko.com/markets/images/52/small/binance.jpg?1706864274',
 'has_trading_incentive': False,
 'trust_score': 10,
 'trust_score_rank': 1,
 'trade_volume_24h_btc': 196325.5893472316,
 'trade_volume_24h_btc_normalized': 107221.49767585589}

In [49]:
# fetch every exchange protocol from defi llama
url = "https://api.llama.fi/protocols"
llama_data = fetch_exchanges_II(url)
# view the first item of the data
llama_data[0]

{'id': '2269',
 'name': 'Binance CEX',
 'address': None,
 'symbol': '-',
 'url': 'https://www.binance.com',
 'description': 'Binance is a cryptocurrency exchange which is the largest exchange in the world in terms of daily trading volume of cryptocurrencies',
 'chain': 'Multi-Chain',
 'logo': 'https://icons.llama.fi/binance-cex.jpg',
 'audits': '0',
 'audit_note': None,
 'gecko_id': None,
 'cmcId': None,
 'category': 'CEX',
 'chains': ['Bitcoin',
  'Ethereum',
  'Binance',
  'Solana',
  'Ripple',
  'Doge',
  'Tron',
  'Base',
  'Arbitrum',
  'Avalanche',
  'Optimism',
  'Litecoin',
  'Polkadot',
  'Hedera',
  'TON',
  'Near',
  'Polygon',
  'Stellar',
  'Algorand',
  'Aptos',
  'Starknet',
  'Chiliz',
  'Manta',
  'Celo',
  'Op_Bnb',
  'Ronin',
  'zkSync Era',
  'Sui'],
 'module': 'binance/index.js',
 'twitter': 'binance',
 'forkedFrom': [],
 'oracles': [],
 'listedAt': 1668170565,
 'methodology': 'We collect the wallets from this binance blog post https://www.binance.com/en/blog/commu

In [50]:
crypto_exchanges = llama_data + gecko_data

# Print the total number of exchanges fetched
print(f"Total exchanges fetched: {len(crypto_exchanges)}")

Total exchanges fetched: 6643


In [51]:
# Create a set to store unique URLs
unique_urls = set()

# Add URLs from cex_data to the set
for exchange in gecko_data:
    # Normalize by stripping trailing slashes
    unique_urls.add(exchange['url'].rstrip('/')) 

# Create a list to store unique exchanges
unique_exchanges = []

# Add exchanges from cex_data to the unique list
for exchange in gecko_data:
    if exchange['url'].rstrip('/') in unique_urls:
        unique_exchanges.append(exchange)

# Add exchanges from ex_data to the unique list
for exchange in llama_data:
    if exchange['url'].rstrip('/') in unique_urls:
        unique_exchanges.append(exchange)

# Print the total number of unique exchanges fetched
# print(f"Total unique exchanges fetched: {len(unique_exchanges)}")

unique_exchanges

[{'id': 'binance',
  'name': 'Binance',
  'year_established': 2017,
  'country': 'Cayman Islands',
  'description': 'One of the world’s largest cryptocurrency exchanges by trading volume, offering a wide range of services including spot, futures, and staking options.',
  'url': 'https://www.binance.com/',
  'image': 'https://coin-images.coingecko.com/markets/images/52/small/binance.jpg?1706864274',
  'has_trading_incentive': False,
  'trust_score': 10,
  'trust_score_rank': 1,
  'trade_volume_24h_btc': 196325.5893472316,
  'trade_volume_24h_btc_normalized': 107221.49767585589},
 {'id': 'bybit_spot',
  'name': 'Bybit',
  'year_established': 2018,
  'country': 'British Virgin Islands',
  'description': 'Bybit is the world’s second-largest cryptocurrency exchange by trading volume, serving a global community of over 60 million users. Founded in 2018, Bybit is redefining openness in the decentralized world by creating a simpler, open and equal ecosystem for everyone. With a strong focus on

In [52]:
with open("legit-exchanges.json", "w") as file:
    json.dump(unique_exchanges, file)

In [53]:
# fetch other legitimate websites asides crypto
# to ensure balance in datasets.
other_websites = "https://raw.githubusercontent.com/seigdev/resources/refs/heads/main/top-websites.csv"

other_raw = fetch_data_csv(other_websites)

others_df = pd.DataFrame(other_raw)

others_df = others_df.drop(columns=0)

others_df = others_df.rename(columns={1: 'url'})

In [54]:

df_raw = pd.read_json("legit-exchanges.json")

# copy the name and urls columns to a new dataframe
crypto_df = df_raw[['url']].copy()

legit_df = pd.concat([crypto_df, others_df], ignore_index=True)

legit_df["url"] = legit_df["url"].apply(format_domain)

# assign string labels to legitimate urls
legit_df.loc[:, 'label'] = 'legit'

# assign string labels to legitimate urls
legit_df.loc[:, 'label_no'] = '0'

# select first 100
legit_df = legit_df[:50000]

legit_df

,url,label,label_no
0,https://www.binance.com/,legit,0
1,https://www.bybit.com,legit,0
2,https://www.okx.com,legit,0
3,https://www.coinbase.com/,legit,0
4,https://www.bitget.com/,legit,0
...,...,...,...
49995,https://ecatholic.com,legit,0
49996,https://reclaim.ai,legit,0
49997,https://gotickets.com,legit,0
49998,https://ctfs.com,legit,0


In [55]:
# url to fetch scam urls from eth-phishing-detect
scam_url = "https://raw.githubusercontent.com/MetaMask/eth-phishing-detect/master/src/config.json"
scam_ex = fetch_exchanges_II(scam_url)

# fetch the list of blacklist urls
blacklist = scam_ex["blacklist"]

In [56]:
blacklist = blacklist_toMap(blacklist)

In [57]:
with open("scam-exchanges.json", "w") as file:
    json.dump(blacklist, file)

In [58]:
blacklist_raw = pd.read_json("scam-exchanges.json")

# copy the name and urls columns to a new dataframe
scam_df = blacklist_raw[['url']].copy()

scam_df["url"] = scam_df["url"].apply(format_domain)

# assign string labels to legitimate urls
scam_df.loc[:, 'label'] = 'scam'

# assign string labels to legitimate urls
scam_df.loc[:, 'label_no'] = '1'

# select first 100
# scam_df = scam_df.sample(n=100000, random_state=42).reset_index(drop=True)
scam_df = scam_df[:100000]

scam_df

,url,label,label_no
0,https://ogntoken-migration.icu,scam,1
1,https://fazla-rabby-rady.github.io,scam,1
2,https://polymaraket.com,scam,1
3,https://predictdex.com,scam,1
4,https://polymarket.mx,scam,1
...,...,...,...
99995,https://blastclaim.pics,scam,1
99996,https://shovel-manta.network,scam,1
99997,https://bouncebit-claim.pages.dev,scam,1
99998,https://arbitrum.wtrust-pad.top,scam,1


In [59]:
# merge both the legit and scam urls together
urls_df = pd.concat([legit_df, scam_df], ignore_index=True)

# shuffle the urls across the dataframe
urls_df = urls_df.sample(frac=1, random_state=42).reset_index(drop=True)

urls_df

,url,label,label_no
0,https://pan-cakev-app.info,scam,1
1,https://harrods.com,legit,0
2,https://claim.dogshouse.store,scam,1
3,https://circle-foundation.com,scam,1
4,https://wepdex.com,scam,1
...,...,...,...
149995,https://hualeebaix.fun,scam,1
149996,https://myetnherwallet.com,scam,1
149997,https://giveawayton.online,scam,1
149998,https://trustwallet-support.tech,scam,1


In [60]:
urls_df.to_json('crypto_data.json', orient='records', lines=False)